# 02 · 微调方法选型：LoRA / QLoRA / AdaLoRA / IA³

四种参数高效方法的取舍，以及怎么生成对应的 PEFT 配置。

## 1. 先看清取舍

| 方法 | 可训练参数 | 显存 | 适合场景 |
|---|---|---|---|
| **LoRA** | ~0.1%~1% | 中 | 通用首选，效果/成本均衡 |
| **QLoRA** | ~0.1%~1%（4-bit 基座） | 低 | 单卡跑大模型（24GB 可上 7B） |
| **AdaLoRA** | ~0.1%~1%（自适应秩） | 中 | 预算固定、希望自动分配秩 |
| **IA³** | ~0.01% | 极低 | 极轻量、多任务快速切换 |

经验起点：**先用 LoRA 打基线**；显存不够换 QLoRA；同等预算下想更好试 AdaLoRA；
只做轻量适配或需要大量任务适配器时试 IA³。

In [ ]:
import sys

sys.path.insert(0, "..")

from llm_finetuning import (
    Config,
    LoRAConfig,
    FinetuningMethod,
)
from llm_finetuning.methods import build_peft_config_dict

for method in (
    FinetuningMethod.LORA,
    FinetuningMethod.QLORA,
    FinetuningMethod.ADALORA,
    FinetuningMethod.IA3,
):
    cfg = Config(model_name_or_path="Qwen/Qwen2.5-7B-Instruct", finetuning_method=method)
    print(f"\n=== {method.value} ===")
    print(f"  默认学习率 : {cfg.training.learning_rate}")
    print(f"  PEFT 配置  : {build_peft_config_dict(cfg)}")

## 2. 关键参数怎么调

- **`r`（秩）**：越大表达力越强，也越容易过拟合。8/16 起步，数据少用小秩。
- **`lora_alpha`**：缩放系数，常用 `2 * r`；实际缩放为 `alpha / r`。
- **`target_modules`**：注意力层 `q/k/v/o` 是基本盘；想更强可加 MLP 的 `gate/up/down`。
- **`lora_dropout`**：数据少时 0.05~0.1 有正则效果。

In [ ]:
# 只微调注意力层（省参数，推荐起步）
attn_only = LoRAConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"])

# 注意力 + MLP（表达力更强，QLoRA 常用）
attn_mlp = LoRAConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

print("attn_only:", attn_only.to_dict())
print("attn_mlp :", attn_mlp.to_dict())

## 3. QLoRA 的 4-bit 细节

QLoRA = 4-bit 量化基座 + LoRA 适配器，三个关键开关：

- `bnb_4bit_quant_type="nf4"`：正态分布权重下优于普通 fp4
- `bnb_4bit_use_double_quant=True`：再省约 0.4GB
- `bnb_4bit_compute_dtype="bfloat16"`：计算时反量化到 bf16

> QLoRA 需要 `bitsandbytes`，且基本只支持 NVIDIA GPU；没有 GPU 请退回 LoRA。

In [ ]:
from llm_finetuning import QLoRAConfig
from llm_finetuning.export import ModelExporter

qlora = QLoRAConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
print(qlora.to_dict())

# 估算不同量化级别下的体积：选型时最实用的一步
for level in ("fp16", "int8", "int4"):
    size = ModelExporter.estimate_size_mb(7_000_000_000, level)
    print(f"7B @ {level:<5} ≈ {size:>7.0f} MB")

## 4. 拿到真正的 PEFT 对象

`build_peft_config_dict` 返回普通字典（**不需要安装 peft**），便于文档生成与测试；
真机训练时用 `build_peft_config`，返回 `peft.LoraConfig` 等真实对象。

In [ ]:
from llm_finetuning.methods import build_peft_config

cfg = Config(
    model_name_or_path="Qwen/Qwen2.5-7B-Instruct",
    finetuning_method=FinetuningMethod.QLORA,
)

try:
    peft_cfg = build_peft_config(cfg)
    print(type(peft_cfg).__name__, "->", peft_cfg)
except ImportError as exc:
    print(f"需要安装 peft 才能生成真实配置对象: {exc}")